# Gold as a Financial Instrument in Egypt
## Fair Value Decomposition, Crisis Analysis & Retail Investor Decision Framework

**DEPI Final Project - Data Analysis Track | Mahmoud Shamoun | 2026**

---

This notebook implements the full production ETL pipeline powering the Gold Egypt analytical platform. It ingests global market data, engineers the local karat price series, decomposes prices into their structural components, computes technical indicators, simulates a friction-adjusted investment portfolio, and loads all results into the live MySQL database.

**Pipeline Stages:**
1. Environment Setup & Configuration
2. **Extract** - Resilient Multi-Source Market Data Ingestion
3. **Transform** - Fair Value Decomposition & Feature Engineering
4. **Transform (Indicators)** - Technical Indicators & Portfolio Simulation
5. **Load** - Idempotent MySQL Database Loading
6. **Visualize** - Crisis Analysis & Structural Break Charts

## Cell 1 - Environment Setup, Logging, and Constants

In [1]:
# ─── Standard Library ────────────────────────────────────────────────────────
import os
import time
import logging
import warnings
from datetime import datetime, timedelta

# ─── Data & Numerics ─────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ─── Market Data ─────────────────────────────────────────────────────────────
import requests
import yfinance as yf

# ─── Database ────────────────────────────────────────────────────────────────
from sqlalchemy import create_engine, text

# ─── Visualisation ───────────────────────────────────────────────────────────
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')

# ─── Production Logging Framework ────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# ─── Domain Constants ────────────────────────────────────────────────────────
START_DATE          = '2020-01-01'
OUNCE_TO_GRAM       = 31.1035
VALUE_BASELINE_EGP  = 15.70          # Jan 2020 EGP/USD anchor (pre-devaluation)

KARAT_FACTORS   = {'24K': 1.000, '21K': 0.875, '18K': 0.750}
MAKING_CHARGES  = {'24K': 60,    '21K': 150,    '18K': 220}     # EGP/gram at entry
SELL_SPREAD     = {'24K': 0.005, '21K': 0.010,  '18K': 0.015}  # bid-ask at liquidation
PORTFOLIO_SEED  = 100_000.0          # EGP starting capital

HTTP_HEADERS = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 Chrome/122.0.0.0 Safari/537.36'
    ),
    'Accept-Language': 'ar,en;q=0.9',
}

CRISIS_EVENTS = {
    '2020-03-15': ('COVID-19 Global Shock',         '#FF6B6B', 'rgba(255,107,107,0.06)'),
    '2022-02-24': ('Russia\u2013Ukraine War',        '#FF9F43', 'rgba(255,159,67,0.06)'),
    '2022-03-21': ('EGP Flotation Event 1',          '#EF476F', 'rgba(239,71,111,0.07)'),
    '2022-10-27': ('EGP Flotation Event 2',          '#EF476F', 'rgba(239,71,111,0.07)'),
    '2023-10-07': ('Gaza Conflict Escalation',       '#FF6B6B', 'rgba(255,107,107,0.07)'),
    '2024-03-06': ('EGP Full Float \u2014 March 2024', '#06D6A0', 'rgba(6,214,160,0.07)'),
    '2024-04-01': ('Gold Hits $2,265/oz',            '#FFD700', 'rgba(255,215,0,0.05)'),
    '2024-09-18': ('Fed Rate Cut Pivot',             '#4CC9F0', 'rgba(76,201,240,0.06)'),
    '2025-01-19': ('Gaza Ceasefire Agreement',       '#06D6A0', 'rgba(6,214,160,0.06)'),
    '2025-04-02': ('Trump Tariff Escalation',        '#FF9F43', 'rgba(255,159,67,0.07)'),
    '2025-04-22': ('Gold Breaks $3,500/oz',          '#FFD700', 'rgba(255,215,0,0.06)'),
    '2025-06-13': ('Iran Strike Event',              '#EF476F', 'rgba(239,71,111,0.08)'),
}

# ── Crisis Events — master dimension (single source of truth for DB + charts) ──
CRISIS_EVENTS_DF = pd.DataFrame([
    ('2020-03-15', 'COVID-19 Global Shock',        'pandemic',        'FF6B6B', 'rgba(255,107,107,0.06)', 'Global COVID-19 pandemic declared. Flight-to-safety drove gold to then-record highs. Local supply chain disruptions amplified domestic price impact.'),
    ('2022-02-24', 'Russia\u2013Ukraine War',      'war_conflict',    'FF9F43', 'rgba(255,159,67,0.06)',  'Russian invasion of Ukraine. Energy price shocks, global inflation surge, and safe-haven demand reinforced upward pressure on global gold prices.'),
    ('2022-03-21', 'EGP Flotation Event 1',         'fx_devaluation',  'EF476F', 'rgba(239,71,111,0.07)', 'CBE first managed devaluation cycle. USD/EGP rate stepped up sharply, transmitting directly into local gold price levels.'),
    ('2022-10-27', 'EGP Flotation Event 2',         'fx_devaluation',  'EF476F', 'rgba(239,71,111,0.07)', 'Second CBE exchange rate adjustment. Further EGP depreciation accelerated local gold price premiums.'),
    ('2023-10-07', 'Gaza Conflict Escalation',      'war_conflict',    'FF6B6B', 'rgba(255,107,107,0.07)', 'Hamas attack on Israel triggers regional conflict escalation. Heightened geopolitical risk premium in gold markets.'),
    ('2024-03-06', 'EGP Full Float \u2014 March 2024', 'fx_devaluation', '06D6A0', 'rgba(6,214,160,0.07)', 'CBE comprehensive exchange rate liberalisation. USD/EGP surged from ~30 to ~50+. Most significant single structural break in the dataset.'),
    ('2024-04-01', 'Gold Hits $2,265/oz',           'price_milestone', 'FFD700', 'rgba(255,215,0,0.05)',  'Global gold spot price reached $2,265/oz \u2014 a then-record high driven by safe-haven demand and Fed rate-cut expectations.'),
    ('2024-09-18', 'Fed Rate Cut Pivot',             'monetary_policy', '4CC9F0', 'rgba(76,201,240,0.06)', 'US Federal Reserve initiates rate-cutting cycle. Lower opportunity cost of holding non-yielding gold provided strong structural support.'),
    ('2025-01-19', 'Gaza Ceasefire Agreement',       'geopolitical',    '06D6A0', 'rgba(6,214,160,0.06)',  'Israel-Hamas ceasefire deal reduces immediate regional risk premium. Gold experienced a brief pullback on reduced safe-haven demand.'),
    ('2025-04-02', 'Trump Tariff Escalation',        'commodity_shock', 'FF9F43', 'rgba(255,159,67,0.07)', 'Sweeping US tariff announcement triggers global trade war fears. Gold surged as investors fled to safe-haven assets.'),
    ('2025-04-22', 'Gold Breaks $3,500/oz',          'price_milestone', 'FFD700', 'rgba(255,215,0,0.06)',  'Gold futures breach $3,500/oz for the first time in history \u2014 driven by tariff shock, USD weakness, and central bank accumulation.'),
    ('2025-06-13', 'Iran Strike Event',              'geopolitical',    'EF476F', 'rgba(239,71,111,0.08)', 'Israeli-US military strikes on Iranian nuclear facilities triggered an immediate geopolitical risk spike, driving gold sharply higher.'),
], columns=['event_date', 'label', 'category', 'color_hex', 'fill_rgba', 'description'])
CRISIS_EVENTS_DF['event_date'] = pd.to_datetime(CRISIS_EVENTS_DF['event_date'])
CRISIS_EVENTS_DF = CRISIS_EVENTS_DF.set_index('event_date')

logger.info('Environment configured. Pipeline constants loaded.')
logger.info('Start date: %s | Baseline EGP/USD: %.2f', START_DATE, VALUE_BASELINE_EGP)

2026-06-22 22:35:55,909 - INFO - Environment configured. Pipeline constants loaded.
2026-06-22 22:35:55,912 - INFO - Start date: 2020-01-01 | Baseline EGP/USD: 15.70


## Cell 2 - Database Connection (.env File Configuration)

All database credentials are loaded at runtime from a local **`.env` file** in the project root directory.  
No credentials are hardcoded in this notebook.

**Before running the Load cell, create a `.env` file in the same directory as this notebook** using the template below:

```dotenv
# .env - Local Environment Secrets
# ⚠️  DO NOT commit this file to version control. Add .env to your .gitignore.

DB_PASSWORD=your_mysql_password_here   # Required
DB_HOST=localhost                       # Optional - default: localhost
DB_PORT=3306                            # Optional - default: 3306
DB_USER=root                            # Optional - default: root
DB_NAME=gold_egypt                      # Optional - default: gold_egypt
```

| Variable | Required | Default | Description |
|---|---|---|---|
| `DB_PASSWORD` | **Required** | *(none)* | MySQL password for the target database |
| `DB_HOST` | Optional | `localhost` | MySQL host address or IP |
| `DB_PORT` | Optional | `3306` | MySQL port number |
| `DB_USER` | Optional | `root` | MySQL username |
| `DB_NAME` | Optional | `gold_egypt` | Target database name |

In [2]:
from dotenv import load_dotenv

# Load environment variables from the local .env file in the project root
load_dotenv()

DB_HOST     = os.getenv('DB_HOST',     'localhost')
DB_PORT     = os.getenv('DB_PORT',     '3306')
DB_USER     = os.getenv('DB_USER',     'root')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_NAME     = os.getenv('DB_NAME',     'gold_egypt')

if not DB_PASSWORD:
    logger.warning(
        'DB_PASSWORD environment variable is not set. '
        'Create a .env file in the project root directory with '
        'DB_PASSWORD=your_password_here before running the Load cell.'
    )

CONNECTION_STRING = (
    f'mysql+pymysql://{DB_USER}:{DB_PASSWORD}'
    f'@{DB_HOST}:{DB_PORT}/{DB_NAME}?charset=utf8mb4'
)

logger.info('Database target: %s@%s:%s/%s', DB_USER, DB_HOST, DB_PORT, DB_NAME)

2026-06-22 22:36:11,225 - INFO - Database target: root@localhost:3306/gold_egypt


---
## Cell 3 - EXTRACT: Resilient Multi-Source Market Data Ingestion

> **Analytical Objective (Section 5 - Data Scope):** The dataset must cover January 2020 to the present day, with all historical series anchored to this pre-shock baseline to capture the full sequence of structural breaks. The pipeline implements a **resilient fallback chain** - GoldPrice.org API → ExchangeRate-API → Frankfurter → yfinance - to ensure high availability even when individual upstream services fail. All data are forward-filled to handle weekend and public holiday gaps, consistent with financial time-series convention.

The end date is kept **open/dynamic**: `datetime.today()` is evaluated at runtime, ensuring the pipeline always pulls the most current data without requiring manual maintenance.

In [3]:
def _scrape_gold_usd_api() -> float | None:
    """Primary live gold spot scraper via GoldPrice.org data API."""
    try:
        response = requests.get(
            'https://data-asg.goldprice.org/dbXRates/USD',
            headers=HTTP_HEADERS, timeout=10
        )
        value = float(response.json()['items'][0]['xauPrice'])
        logger.info('Gold USD scraped via GoldPrice.org API: %.4f', value)
        return value
    except Exception as exc:
        logger.warning('GoldPrice.org scraper failed: %s', exc)
        return None


def _scrape_gold_usd_yf() -> float | None:
    """Fallback: live gold spot via yfinance GC=F futures."""
    try:
        df = yf.download('GC=F', period='5d', progress=False, auto_adjust=False)
        if df.empty:
            return None
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.droplevel(1)
        col = 'Adj Close' if 'Adj Close' in df.columns else 'Close'
        value = float(df[col].dropna().iloc[-1])
        logger.info('Gold USD scraped via yfinance GC=F: %.4f', value)
        return value
    except Exception as exc:
        logger.warning('yfinance GC=F scraper failed: %s', exc)
        return None


def _scrape_usd_egp_primary() -> float | None:
    """Primary USD/EGP rate via ExchangeRate-API."""
    try:
        response = requests.get(
            'https://api.exchangerate-api.com/v4/latest/USD',
            headers=HTTP_HEADERS, timeout=10
        )
        value = float(response.json()['rates']['EGP'])
        logger.info('USD/EGP scraped via ExchangeRate-API: %.4f', value)
        return value
    except Exception as exc:
        logger.warning('ExchangeRate-API scraper failed: %s', exc)
        return None


def _scrape_usd_egp_backup() -> float | None:
    """Backup USD/EGP rate via Frankfurter API."""
    try:
        response = requests.get(
            'https://api.frankfurter.app/latest?from=USD&to=EGP',
            headers=HTTP_HEADERS, timeout=10
        )
        value = float(response.json()['rates']['EGP'])
        logger.info('USD/EGP scraped via Frankfurter: %.4f', value)
        return value
    except Exception as exc:
        logger.warning('Frankfurter scraper failed: %s', exc)
        return None


def _scrape_usd_egp_yf() -> float | None:
    """Final fallback USD/EGP rate via yfinance EGP=X ticker."""
    try:
        df = yf.download('EGP=X', period='5d', progress=False, auto_adjust=False)
        if df.empty:
            return None
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.droplevel(1)
        col = 'Adj Close' if 'Adj Close' in df.columns else 'Close'
        value = float(df[col].dropna().iloc[-1])
        logger.info('USD/EGP scraped via yfinance EGP=X: %.4f', value)
        return value
    except Exception as exc:
        logger.warning('yfinance EGP=X scraper failed: %s', exc)
        return None


def _download_yf_series(ticker: str, name: str, start: str, end: str) -> pd.DataFrame:
    """Download a single yfinance series with retry logic. Returns a single-column DataFrame."""
    for attempt in range(3):
        try:
            df = yf.download(ticker, start=start, end=end,
                             progress=False, auto_adjust=False)
            if df.empty:
                logger.warning('Empty response for ticker %s on attempt %d', ticker, attempt + 1)
                break
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.droplevel(1)
            col = 'Adj Close' if 'Adj Close' in df.columns else 'Close'
            series = df[[col]].rename(columns={col: name})
            logger.info('Downloaded %s (%s): %d rows', name, ticker, len(series))
            return series
        except Exception as exc:
            logger.warning('yfinance download failed for %s (attempt %d): %s', ticker, attempt + 1, exc)
            if attempt < 2:
                time.sleep(2)
    return pd.DataFrame()


def extract_market_data(start: str = START_DATE) -> pd.DataFrame:
    """
    PRIMARY EXTRACTION FUNCTION.

    Downloads all five market series from Yahoo Finance from `start` to the
    current date (open end - evaluated dynamically at runtime). Applies
    forward-fill to handle weekend/holiday gaps, then drops rows with any
    remaining NaN to ensure clean insertion.

    Returns
    -------
    pd.DataFrame with DatetimeIndex and columns:
        gold_usd_oz, usd_egp_official, crude_oil, us_10y_treasury, sp500
    """
    end_date = datetime.today().strftime('%Y-%m-%d')
    logger.info('EXTRACT | Pulling market data from %s to %s (dynamic end date)', start, end_date)

    ticker_map = {
        'gold_usd_oz':       'GC=F',
        'usd_egp_official':  'EGP=X',
        'crude_oil':         'CL=F',
        'us_10y_treasury':   '^TNX',
        'sp500':             '^GSPC',
    }

    frames = []
    for col_name, ticker in ticker_map.items():
        series_df = _download_yf_series(ticker, col_name, start, end_date)
        if not series_df.empty:
            frames.append(series_df)

    if not frames:
        logger.error('EXTRACT | All yfinance downloads returned empty. Cannot proceed.')
        raise RuntimeError('Extraction failed: no data returned from any ticker.')

    raw = pd.concat(frames, axis=1, sort=True)
    raw.index = pd.to_datetime(raw.index)
    raw.ffill(inplace=True)
    raw.bfill(inplace=True)
    raw.dropna(inplace=True)

    logger.info(
        'EXTRACT | Complete. Shape: %s | Date range: %s \u2192 %s',
        raw.shape,
        raw.index.min().strftime('%Y-%m-%d'),
        raw.index.max().strftime('%Y-%m-%d')
    )
    return raw


# ─── Execute extraction ───────────────────────────────────────────────────────
df_raw = extract_market_data()
print(df_raw.tail())
print(f'\nTotal trading days extracted: {len(df_raw):,}')

2026-06-22 22:36:41,792 - INFO - EXTRACT | Pulling market data from 2020-01-01 to 2026-06-22 (dynamic end date)
2026-06-22 22:36:43,249 - INFO - Downloaded gold_usd_oz (GC=F): 1626 rows
2026-06-22 22:36:43,865 - INFO - Downloaded usd_egp_official (EGP=X): 1683 rows
2026-06-22 22:36:44,589 - INFO - Downloaded crude_oil (CL=F): 1626 rows
2026-06-22 22:36:45,300 - INFO - Downloaded us_10y_treasury (^TNX): 1625 rows
2026-06-22 22:36:45,864 - INFO - Downloaded sp500 (^GSPC): 1624 rows
2026-06-22 22:36:45,878 - INFO - EXTRACT | Complete. Shape: (1684, 5) | Date range: 2020-01-01 → 2026-06-19


Price       gold_usd_oz  usd_egp_official  crude_oil  us_10y_treasury  \
Date                                                                    
2026-06-15  4328.000000         51.386398  80.750000            4.469   
2026-06-16  4330.899902         50.319302  76.050003            4.428   
2026-06-17  4358.899902         50.091301  76.790001            4.463   
2026-06-18  4224.100098         49.895000  76.599998            4.451   
2026-06-19  4224.100098         49.894833  76.599998            4.451   

Price             sp500  
Date                     
2026-06-15  7554.290039  
2026-06-16  7511.350098  
2026-06-17  7420.100098  
2026-06-18  7500.580078  
2026-06-19  7500.580078  

Total trading days extracted: 1,684


---
## Cell 4 - TRANSFORM: Fair Value Decomposition & Price Anatomy

> **Core Research Question (Section 3 - Research Questions):** *"What fraction of Egyptian gold price movement is attributable to global spot price appreciation versus USD/EGP exchange rate depreciation versus local market premium expansion?"* This transformation answers this question directly by engineering the three-component **Price Anatomy Framework**:
>
> 1. **Value-Driven Component** - what the gram would cost at the pre-devaluation Jan 2020 EGP/USD anchor of 15.70
> 2. **Currency & Inflation Premium** - the residual reflecting EGP weakness and local market distortions
> 3. **PremPct%** - the fraction of the current price that is "pure" inflation/devaluation premium
>
> A **Fair Value Index (FVI = Actual / Theoretical)** above 1.0 signals the local market is trading at a speculative or illiquidity premium above theoretical parity.

**Core Equations (Section 6.1):**
```
Theoretical_24K  = (Gold_USD_oz / 31.1035) × USD_EGP_Rate
Theoretical_Karat = Theoretical_24K × Karat_Factor
Value_Driven     = (Gold_USD_oz / 31.1035) × 15.70   ← Jan 2020 baseline anchor
Inflation_Premium = Actual_Price − Value_Driven
PremPct%         = (Inflation_Premium / Actual_Price) × 100
FVI              = Actual_Price / Theoretical_Karat
```

In [4]:
def transform_prices(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    PRICE ANATOMY TRANSFORMATION.

    Produces two output DataFrames aligned to the same DatetimeIndex:
      - gold_prices_df : cleaned global market baselines (matches gold_prices table)
      - karat_prices_df: engineered karat columns with decomposition (matches karat_prices table)

    Safety: crude_oil values <= 0 are replaced with NaN and forward/back-filled
    before loading to prevent violations of the MySQL CHECK (crude_oil > 0) constraint.
    """
    logger.info('TRANSFORM | Starting price anatomy transformation.')

    work = df.copy()

    # ── Data Validation: crude_oil CHECK constraint guard ──────────────────────
    # Yahoo Finance occasionally returns values <= 0 for CL=F during monthly
    # contract rollovers. Replace and interpolate to satisfy CHECK (crude_oil > 0).
    invalid_oil_mask = work['crude_oil'] <= 0
    n_invalid = invalid_oil_mask.sum()
    if n_invalid > 0:
        logger.warning(
            'TRANSFORM | Found %d crude_oil values <= 0 (contract rollover artefacts). '
            'Replacing with NaN and applying ffill/bfill.', n_invalid
        )
        work.loc[invalid_oil_mask, 'crude_oil'] = np.nan
        work['crude_oil'] = work['crude_oil'].ffill().bfill()
    else:
        logger.info('TRANSFORM | crude_oil validation passed: no zero/negative values found.')

    # ── Gold Prices (fact table) ───────────────────────────────────────────────
    gold_prices_df = work[[
        'gold_usd_oz', 'usd_egp_official', 'crude_oil',
        'us_10y_treasury', 'sp500'
    ]].copy()
    gold_prices_df.index.name = 'trade_date'

    # ── Theoretical 24K Base (EGP/gram) ───────────────────────────────────────
    theoretical_24k = (work['gold_usd_oz'] / OUNCE_TO_GRAM) * work['usd_egp_official']

    # ── Karat Price Engineering ────────────────────────────────────────────────
    karat_rows = {}

    for karat, factor in KARAT_FACTORS.items():
        karat_key = karat.lower().replace('k', '')   # '24K' → '24'

        # Theoretical karat price: what the market should price this gram at
        theoretical_karat = theoretical_24k * factor

        # Value-driven component: using Jan 2020 baseline EGP/USD (15.70)
        # This isolates the pure global commodity appreciation.
        value_driven = (work['gold_usd_oz'] / OUNCE_TO_GRAM) * factor * VALUE_BASELINE_EGP

        # Inflation & currency premium: the residual revealing local EGP weakness
        infl_prem = theoretical_karat - value_driven

        # Premium as a percentage of the actual theoretical price
        prem_pct = (infl_prem / theoretical_karat) * 100

        # price_{karat_key}k = retail BUY price (spot + making charge)
        # theoretical_{karat_key}k = pure spot price (for FVI reference & decomposition)
        karat_rows[f'price_{karat_key}k']        = theoretical_karat + MAKING_CHARGES[karat]
        karat_rows[f'theoretical_{karat_key}k']  = theoretical_24k if karat == '24K' else theoretical_karat
        karat_rows[f'value_driven_{karat_key}k'] = value_driven
        karat_rows[f'infl_prem_{karat_key}k']    = infl_prem
        karat_rows[f'prem_pct_{karat_key}k']     = prem_pct

    karat_prices_df = pd.DataFrame(karat_rows, index=work.index)
    karat_prices_df.index.name = 'trade_date'

    # ── Schema alignment: keep only columns present in the karat_prices table ─
    karat_schema_cols = [
        'price_24k', 'price_21k', 'price_18k',
        'theoretical_24k',
        'value_driven_24k', 'infl_prem_24k', 'prem_pct_24k',
    ]
    karat_prices_df = karat_prices_df[karat_schema_cols]

    logger.info(
        'TRANSFORM | gold_prices: %d rows | karat_prices: %d rows',
        len(gold_prices_df), len(karat_prices_df)
    )
    logger.info(
        'TRANSFORM | 24K price range: %.0f \u2013 %.0f EGP/gram',
        karat_prices_df['price_24k'].min(),
        karat_prices_df['price_24k'].max()
    )
    logger.info(
        'TRANSFORM | Avg inflation premium (24K): %.1f%%',
        karat_prices_df['prem_pct_24k'].mean()
    )

    return gold_prices_df, karat_prices_df


# ─── Execute transformation ───────────────────────────────────────────────────
df_gold_prices, df_karat_prices = transform_prices(df_raw)

print('Gold Prices (last 5 rows):')
print(df_gold_prices.tail())
print('\nKarat Prices (last 5 rows):')
print(df_karat_prices.tail())

2026-06-22 22:37:42,523 - INFO - TRANSFORM | Starting price anatomy transformation.
2026-06-22 22:37:42,525 - WARNING - TRANSFORM | Found 1 crude_oil values <= 0 (contract rollover artefacts). Replacing with NaN and applying ffill/bfill.
2026-06-22 22:37:42,538 - INFO - TRANSFORM | gold_prices: 1684 rows | karat_prices: 1684 rows
2026-06-22 22:37:42,541 - INFO - TRANSFORM | 24K price range: 806 – 8930 EGP/gram
2026-06-22 22:37:42,542 - INFO - TRANSFORM | Avg inflation premium (24K): 35.5%


Gold Prices (last 5 rows):
Price       gold_usd_oz  usd_egp_official  crude_oil  us_10y_treasury  \
trade_date                                                              
2026-06-15  4328.000000         51.386398  80.750000            4.469   
2026-06-16  4330.899902         50.319302  76.050003            4.428   
2026-06-17  4358.899902         50.091301  76.790001            4.463   
2026-06-18  4224.100098         49.895000  76.599998            4.451   
2026-06-19  4224.100098         49.894833  76.599998            4.451   

Price             sp500  
trade_date               
2026-06-15  7554.290039  
2026-06-16  7511.350098  
2026-06-17  7420.100098  
2026-06-18  7500.580078  
2026-06-19  7500.580078  

Karat Prices (last 5 rows):
              price_24k    price_21k    price_18k  theoretical_24k  \
trade_date                                                           
2026-06-15  7210.331375  6406.539953  5582.748531      7150.331375   
2026-06-16  7066.538120  6280.720855  54

---
## Cell 5 - TRANSFORM: Technical Indicators & Portfolio Simulation

> **Retail Investor Decision Framework (Section 4.3 & 4.4 - Objectives):** Egyptian retail investors lack access to institutional-grade quantitative decision tools. This transformation addresses the core research question: *"Which automated technical signal best characterises the current entry/exit opportunity across gold karat grades?"* and *"How do gold returns (net of making charges and bid-ask spreads) compare to USD and EGP cash positions since January 2020?"*
>
> **Technical Indicators Computed:**
> - **SMA-50 & SMA-200** - trend direction and Golden Cross / Death Cross detection
> - **MACD (12, 26, 9)** - momentum crossover signal engine
> - **RSI-14** - overbought (>70) / oversold (<30) momentum oscillator
> - **Bollinger Bands (20-day, 2σ)** - volatility-adjusted range classifier
> - **Composite BUY / HOLD / SELL signal** - synthesised from RSI + MACD crossover logic
>
> **Portfolio Simulation Frictions (Section 6.3):**
> - Entry: `Raw_Price + Making_Charge` (24K: +60 EGP/gram, 21K: +150 EGP/gram, 18K: +220 EGP/gram)
> - Exit: `Portfolio_Value × (1 − Sell_Spread)` (24K: 0.5%, 21K: 1.0%, 18K: 1.5%)

In [5]:
def compute_technical_indicators(df_gold: pd.DataFrame, df_karat: pd.DataFrame) -> pd.DataFrame:
    """
    TECHNICAL INDICATORS TRANSFORMATION.

    Computes all technical indicators for 24K (the schema stores only 24K
    indicators per the technical_indicators table definition). Returns a
    DataFrame matching the technical_indicators table schema.
    """
    logger.info('TRANSFORM | Computing technical indicators for 24K.')

    price = df_karat['price_24k'].copy()

    rows = {}

    # ── 24K Indicators ────────────────────────────────────────────────────────

    # Simple Moving Averages
    rows['sma50_24k']  = price.rolling(50,  min_periods=1).mean()
    rows['sma200_24k'] = price.rolling(200, min_periods=1).mean()

    # RSI-14
    delta = price.diff()
    gain  = delta.where(delta > 0, 0.0).rolling(14).mean()
    loss  = (-delta.where(delta < 0, 0.0)).rolling(14).mean()
    rs    = gain / loss.replace(0, np.nan)
    rsi   = 100 - (100 / (1 + rs))
    rows['rsi_24k'] = rsi

    # MACD (12, 26, 9)
    ema12    = price.ewm(span=12, adjust=False).mean()
    ema26    = price.ewm(span=26, adjust=False).mean()
    macd     = ema12 - ema26
    macd_sig = macd.ewm(span=9, adjust=False).mean()
    rows['macd_24k']      = macd
    rows['macd_sig_24k']  = macd_sig
    rows['macd_hist_24k'] = macd - macd_sig

    # Bollinger Bands (20-day, 2σ)
    bb_mid = price.rolling(20).mean()
    bb_std = price.rolling(20).std()
    rows['bb_upper_24k'] = bb_mid + 2 * bb_std
    rows['bb_mid_24k']   = bb_mid
    rows['bb_lower_24k'] = bb_mid - 2 * bb_std

    # Composite BUY / SELL / HOLD signal
    buy_cond  = (rsi < 30) | (
        (macd > macd_sig) & (macd.shift(1) <= macd_sig.shift(1))
    )
    sell_cond = (rsi > 70) | (
        (macd < macd_sig) & (macd.shift(1) >= macd_sig.shift(1))
    )
    rows['signal_24k'] = np.select(
        [buy_cond, sell_cond], ['BUY', 'SELL'], default='HOLD'
    )

    tech_df = pd.DataFrame(rows, index=df_karat.index)
    tech_df.index.name = 'trade_date'

    latest_signal = tech_df['signal_24k'].iloc[-1]
    latest_rsi    = tech_df['rsi_24k'].iloc[-1]
    logger.info(
        'TRANSFORM | Technical indicators complete. '
        'Latest 24K signal: %s | RSI: %.1f',
        latest_signal, latest_rsi
    )

    return tech_df


def compute_portfolio_simulation(
    df_gold: pd.DataFrame,
    df_karat: pd.DataFrame
) -> pd.DataFrame:
    """
    NET-RETURN INVESTMENT PORTFOLIO SIMULATION.

    Simulates a 100,000 EGP investment from the first available date across
    five asset classes: gold 24K, 21K, 18K, USD, and EGP cash.

    Transaction frictions applied:
      - Entry: Raw_Price_per_gram + Making_Charge (المصنعية)
      - Exit:  Portfolio_Value × (1 - Sell_Spread)  (bid-ask spread)

    Returns a DataFrame matching the portfolio_simulation table schema.
    """
    logger.info('TRANSFORM | Computing friction-adjusted portfolio simulation.')

    rows = {}

    # ── 24K Gold Portfolio ────────────────────────────────────────────────────
    price_24k     = df_karat['price_24k']
    # price_24k already includes MAKING_CHARGES['24K'] — no double-counting
    entry_24k     = price_24k.iloc[0]
    grams_24k     = PORTFOLIO_SEED / entry_24k
    net_port_24k  = (price_24k * grams_24k) * (1 - SELL_SPREAD['24K'])
    cum_ret_24k   = (net_port_24k / PORTFOLIO_SEED) - 1
    peak_24k      = net_port_24k.cummax()
    drawdown_24k  = (net_port_24k / peak_24k) - 1

    rows['port_24k_net']   = net_port_24k
    rows['cum_return_24k'] = cum_ret_24k
    rows['drawdown_24k']   = drawdown_24k

    # ── 21K Gold Portfolio ────────────────────────────────────────────────────
    price_21k     = df_karat['price_21k']
    # price_21k already includes MAKING_CHARGES['21K'] — no double-counting
    entry_21k     = price_21k.iloc[0]
    grams_21k     = PORTFOLIO_SEED / entry_21k
    net_port_21k  = (price_21k * grams_21k) * (1 - SELL_SPREAD['21K'])
    cum_ret_21k   = (net_port_21k / PORTFOLIO_SEED) - 1
    peak_21k      = net_port_21k.cummax()
    drawdown_21k  = (net_port_21k / peak_21k) - 1

    rows['port_21k_net']   = net_port_21k
    rows['cum_return_21k'] = cum_ret_21k
    rows['drawdown_21k']   = drawdown_21k

    # ── 18K Gold Portfolio ────────────────────────────────────────────────────
    price_18k     = df_karat['price_18k']
    # price_18k already includes MAKING_CHARGES['18K'] — no double-counting
    entry_18k     = price_18k.iloc[0]
    grams_18k     = PORTFOLIO_SEED / entry_18k
    net_port_18k  = (price_18k * grams_18k) * (1 - SELL_SPREAD['18K'])
    cum_ret_18k   = (net_port_18k / PORTFOLIO_SEED) - 1
    peak_18k      = net_port_18k.cummax()
    drawdown_18k  = (net_port_18k / peak_18k) - 1

    rows['port_18k_net']   = net_port_18k
    rows['cum_return_18k'] = cum_ret_18k
    rows['drawdown_18k']   = drawdown_18k

    # ── USD Portfolio ─────────────────────────────────────────────────────────
    usd_rate_0  = df_gold['usd_egp_official'].iloc[0]
    usd_bought  = PORTFOLIO_SEED / usd_rate_0
    port_usd    = usd_bought * df_gold['usd_egp_official']

    rows['port_usd']  = port_usd

    # ── EGP Cash (baseline) ───────────────────────────────────────────────────
    rows['port_cash'] = PORTFOLIO_SEED

    port_df = pd.DataFrame(rows, index=df_karat.index)
    port_df.index.name = 'trade_date'

    # ── Validation: enforce drawdown <= 0 constraint ──────────────────────────
    for col in ['drawdown_24k', 'drawdown_21k', 'drawdown_18k']:
        pos_count = (port_df[col] > 0).sum()
        if pos_count > 0:
            logger.warning(
                'TRANSFORM | %d positive drawdown values in %s - clamping to 0.',
                pos_count, col
            )
            port_df[col] = port_df[col].clip(upper=0.0)

    final_ret_24k = cum_ret_24k.iloc[-1] * 100
    final_ret_21k = cum_ret_21k.iloc[-1] * 100
    final_ret_18k = cum_ret_18k.iloc[-1] * 100
    usd_total_ret = ((port_usd.iloc[-1] / PORTFOLIO_SEED) - 1) * 100

    logger.info(
        'TRANSFORM | Portfolio simulation complete.\n'
        '  24K net return: %+.1f%%\n'
        '  21K net return: %+.1f%%\n'
        '  18K net return: %+.1f%%\n'
        '  USD return:     %+.1f%%',
        final_ret_24k, final_ret_21k, final_ret_18k, usd_total_ret
    )

    return port_df


# ─── Execute transformations ──────────────────────────────────────────────────
df_technical  = compute_technical_indicators(df_gold_prices, df_karat_prices)
df_portfolio  = compute_portfolio_simulation(df_gold_prices, df_karat_prices)

print('Technical Indicators (last 5 rows):')
print(df_technical[['sma50_24k', 'rsi_24k', 'macd_24k', 'signal_24k']].tail())
print('\nPortfolio Simulation (last 5 rows):')
print(df_portfolio[['port_24k_net', 'port_21k_net', 'port_18k_net', 'port_usd']].tail())

2026-06-22 22:37:50,732 - INFO - TRANSFORM | Computing technical indicators for 24K.
2026-06-22 22:37:50,747 - INFO - TRANSFORM | Technical indicators complete. Latest 24K signal: BUY | RSI: 27.8
2026-06-22 22:37:50,749 - INFO - TRANSFORM | Computing friction-adjusted portfolio simulation.
2026-06-22 22:37:50,762 - INFO - TRANSFORM | Portfolio simulation complete.
  24K net return: +705.1%
  21K net return: +619.3%
  18K net return: +545.9%
  USD return:     +211.6%


Technical Indicators (last 5 rows):
              sma50_24k    rsi_24k    macd_24k signal_24k
trade_date                                               
2026-06-15  7806.299914  38.180506 -218.392380       HOLD
2026-06-16  7784.086265  36.982301 -218.494890       HOLD
2026-06-17  7757.571396  34.285695 -215.020593       HOLD
2026-06-18  7729.065961  25.999173 -229.292664        BUY
2026-06-19  7702.109920  27.800201 -237.863283        BUY

Portfolio Simulation (last 5 rows):
             port_24k_net   port_21k_net   port_18k_net       port_usd
trade_date                                                            
2026-06-15  849205.027723  758016.224164  680050.679968  320918.388375
2026-06-16  832269.612609  743129.418139  666913.777038  314254.154887
2026-06-17  833841.457432  744511.123213  668133.066482  312830.245047
2026-06-18  805133.475138  719275.830136  645864.112684  311604.308918
2026-06-19  805130.790442  719273.470198  645862.030150  311603.260682


---
## Cell 6 - LOAD: Idempotent MySQL Database Loading

> **Production Loading Protocol:** The destination MySQL schema and all tables are **already created and live**. This cell performs no DDL (no CREATE TABLE statements). Instead, it executes a strict **Idempotent Load** pattern:
>
> 1. Open a single `engine.begin()` transaction block for atomicity
> 2. Execute explicit `DELETE FROM` statements in **reverse dependency order** (child tables first) to prevent Foreign Key constraint violations
> 3. Append all fresh DataFrames using `.to_sql(if_exists='append')` in **forward dependency order** (parent table first)
>
> This guarantees the pipeline is safe to re-run at any time without producing duplicate rows or orphaned foreign key records.

In [6]:
def load_to_mysql(
    gold_prices_df:   pd.DataFrame,
    karat_prices_df:  pd.DataFrame,
    technical_df:     pd.DataFrame,
    portfolio_df:     pd.DataFrame,
    connection_string: str
) -> None:
    """
    IDEMPOTENT MYSQL LOAD FUNCTION.

    Cleans all target tables (reverse FK order) and reloads with fresh data
    (forward FK order) within a single atomic transaction. No DDL is executed.

    Delete order  (child → parent):  portfolio → technical → karat → gold_prices
    Insert order  (parent → child):  gold_prices → karat → technical → portfolio
    """
    logger.info('LOAD | Creating SQLAlchemy engine for target database.')
    engine = create_engine(connection_string, pool_pre_ping=True)

    logger.info('LOAD | Beginning atomic transaction.')
    with engine.begin() as conn:

        # ── Step 1: Clean in reverse FK dependency order ──────────────────────
        logger.info('LOAD | Deleting child tables (reverse FK order).')
        conn.execute(text('DELETE FROM crisis_events'))      # standalone — no FK deps
        conn.execute(text('DELETE FROM portfolio_simulation'))
        conn.execute(text('DELETE FROM technical_indicators'))
        conn.execute(text('DELETE FROM karat_prices'))
        conn.execute(text('DELETE FROM gold_prices'))
        logger.info('LOAD | All target tables cleared successfully.')

        # ── Step 2: Insert in forward FK dependency order ─────────────────────

        # 2a. gold_prices (parent table - no FK dependencies)
        gold_prices_df.to_sql(
            name='gold_prices',
            con=conn,
            if_exists='append',
            index=True,
            index_label='trade_date',
            method='multi',
            chunksize=500
        )
        logger.info('LOAD | gold_prices: %d rows inserted.', len(gold_prices_df))

        # 2b. karat_prices (FK → gold_prices.trade_date)
        karat_prices_df.to_sql(
            name='karat_prices',
            con=conn,
            if_exists='append',
            index=True,
            index_label='trade_date',
            method='multi',
            chunksize=500
        )
        logger.info('LOAD | karat_prices: %d rows inserted.', len(karat_prices_df))

        # 2c. technical_indicators (FK → gold_prices.trade_date)
        technical_df.to_sql(
            name='technical_indicators',
            con=conn,
            if_exists='append',
            index=True,
            index_label='trade_date',
            method='multi',
            chunksize=500
        )
        logger.info('LOAD | technical_indicators: %d rows inserted.', len(technical_df))

        # 2d. portfolio_simulation (FK → gold_prices.trade_date)
        portfolio_df.to_sql(
            name='portfolio_simulation',
            con=conn,
            if_exists='append',
            index=True,
            index_label='trade_date',
            method='multi',
            chunksize=500
        )
        logger.info('LOAD | portfolio_simulation: %d rows inserted.', len(portfolio_df))

        # 2e. crisis_events (standalone dimension — no FK constraints)
        CRISIS_EVENTS_DF.to_sql(
            name='crisis_events',
            con=conn,
            if_exists='append',
            index=True,
            index_label='event_date',
            method='multi',
            chunksize=500
        )
        logger.info('LOAD | crisis_events: %d rows inserted.', len(CRISIS_EVENTS_DF))

    logger.info('LOAD | Transaction committed successfully. All 5 tables refreshed.')
    engine.dispose()


# ─── Execute load ─────────────────────────────────────────────────────────────
# Uncomment the call below after configuring your .env file.

load_to_mysql(
    gold_prices_df=df_gold_prices,
    karat_prices_df=df_karat_prices,
    technical_df=df_technical,
    portfolio_df=df_portfolio,
    connection_string=CONNECTION_STRING
)

logger.info('LOAD | Cell executed. Uncomment load_to_mysql() to activate database write.')

2026-06-22 22:38:18,959 - INFO - LOAD | Creating SQLAlchemy engine for target database.
2026-06-22 22:38:19,167 - INFO - LOAD | Beginning atomic transaction.
2026-06-22 22:38:20,599 - INFO - LOAD | Deleting child tables (reverse FK order).
2026-06-22 22:38:21,590 - INFO - LOAD | All target tables cleared successfully.
2026-06-22 22:38:21,982 - INFO - LOAD | gold_prices: 1684 rows inserted.
2026-06-22 22:38:22,532 - INFO - LOAD | karat_prices: 1684 rows inserted.
2026-06-22 22:38:22,996 - INFO - LOAD | technical_indicators: 1684 rows inserted.
2026-06-22 22:38:23,514 - INFO - LOAD | portfolio_simulation: 1684 rows inserted.
2026-06-22 22:38:23,549 - INFO - LOAD | crisis_events: 12 rows inserted.
2026-06-22 22:38:23,573 - INFO - LOAD | Transaction committed successfully. All 5 tables refreshed.
2026-06-22 22:38:23,577 - INFO - LOAD | Cell executed. Uncomment load_to_mysql() to activate database write.


---
## Cell 7 - VISUALIZE: Crisis Analysis & Structural Breaks

> **Analytical Objective (Section 4.1 - Macroeconomic Relationship & Crisis Window Analysis):** Egypt's gold market has been subjected to an extraordinary sequence of structural shocks since January 2020. Each shock leaves a measurable imprint on price levels and on the composition of the inflation premium. This visualization section implements the **Crisis Window Event Study** - overlaying geopolitical and macroeconomic events onto the price series to identify structural breaks and regime shifts. Key questions addressed:
>
> - **COVID-19 (March 2020):** Did flight-to-safety drive the global component or the local premium?
> - **EGP Flotation Cycles (2022, 2024):** How sharply did devaluation events transmit into local gold prices?
> - **Trump Tariff Shock & $3,500/oz (April 2025):** Does the data show a permanent regime shift or a transient spike?

All charts use the same Plotly dark theme as the Streamlit application for visual consistency across all deliverable layers.

In [7]:
# ─── Shared Plotly Dark Theme Configuration ───────────────────────────────────

PLOTLY_BASE = dict(
    template='plotly_dark',
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(10,21,37,0.7)',
    font=dict(family='Arial, sans-serif', color='#8D99AE', size=11),
    hovermode='x unified',
    margin=dict(l=60, r=30, t=80, b=60),
    hoverlabel=dict(bgcolor='#0C1830', font_size=12, bordercolor='#1E3A5F'),
)


def add_crisis_overlays(fig, data_index, rows=None, y_ann=0.97):
    """Add vertical lines and annotations for all crisis events to a Plotly figure."""
    for date_str, (label, color, fill) in sorted(CRISIS_EVENTS.items()):
        dt = pd.to_datetime(date_str)
        if dt < data_index.min():
            continue
        x1 = (dt + timedelta(days=15)).strftime('%Y-%m-%d')
        rect_kw = dict(
            x0=date_str, x1=x1, fillcolor=fill,
            opacity=1, layer='below', line_width=0
        )
        if rows:
            for r in rows:
                fig.add_vrect(**rect_kw, row=r, col=1)
        else:
            fig.add_vrect(**rect_kw)
        fig.add_vline(
            x=pd.to_datetime(date_str).timestamp() * 1000,
            line=dict(color=color, width=0.9, dash='dot'), opacity=0.60
        )
        fig.add_annotation(
            x=date_str, y=y_ann, yref='paper', xref='x',
            text=label, showarrow=False,
            font=dict(size=8, color=color, family='Arial'),
            textangle=-90, xanchor='left', yanchor='top',
            bgcolor='rgba(3,6,15,0.55)', borderpad=2
        )

In [8]:
# ─── Chart 1: Gold Price Track with SMA Overlays ─────────────────────────────

fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=df_karat_prices.index, y=df_karat_prices['price_24k'],
    name='24K EGP/gram',
    line=dict(color='#FFD700', width=2),
    fill='tozeroy', fillcolor='rgba(255,215,0,0.06)',
    hovertemplate='%{x|%d %b %Y}<br>%{y:,.0f} EGP/gram<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=df_technical.index, y=df_technical['sma50_24k'],
    name='SMA-50', line=dict(color='#FF9F43', width=1.3, dash='dot'),
    hovertemplate='SMA50: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=df_technical.index, y=df_technical['sma200_24k'],
    name='SMA-200', line=dict(color='#A855F7', width=1.3, dash='dot'),
    hovertemplate='SMA200: %{y:,.0f}<extra></extra>'
))

add_crisis_overlays(fig1, df_karat_prices.index)

fig1.update_layout(
    **PLOTLY_BASE,
    height=520,
    title=dict(
        text='Egyptian Gold Price (24K EGP/gram) \u2014 Crisis Overlay & Structural Breaks | Jan 2020\u2013Present',
        font=dict(size=13, color='#FFD700'), x=0.5
    ),
    xaxis=dict(title='Date', gridcolor='rgba(19,34,56,0.8)', zeroline=False),
    yaxis=dict(title='EGP per gram', gridcolor='rgba(19,34,56,0.8)', zeroline=False),
    legend=dict(orientation='h', y=1.08, x=0.5, xanchor='center', bgcolor='rgba(0,0,0,0)')
)

fig1.show()

In [9]:
# ─── Chart 2: Price Anatomy - Stacked Area (Value vs Premium) ────────────────

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=df_karat_prices.index, y=df_karat_prices['value_driven_24k'],
    name='Global Value Component (Jan 2020 FX Anchor)',
    stackgroup='anatomy', mode='lines',
    line=dict(width=0), fillcolor='rgba(255,249,196,0.65)',
    hovertemplate='Value-Driven: %{y:,.0f} EGP<extra></extra>'
))

fig2.add_trace(go.Scatter(
    x=df_karat_prices.index, y=df_karat_prices['infl_prem_24k'],
    name='Currency & Inflation Premium',
    stackgroup='anatomy', mode='lines',
    line=dict(width=0), fillcolor='rgba(239,71,111,0.55)',
    hovertemplate='Inflation Premium: %{y:,.0f} EGP<extra></extra>'
))

add_crisis_overlays(fig2, df_karat_prices.index)

fig2.update_layout(
    **PLOTLY_BASE,
    height=500,
    title=dict(
        text='Fair Value Decomposition \u2014 24K | Global Value vs Currency & Inflation Premium',
        font=dict(size=13, color='#FFD700'), x=0.5
    ),
    xaxis=dict(title='Date', gridcolor='rgba(19,34,56,0.8)', zeroline=False),
    yaxis=dict(title='EGP per gram', gridcolor='rgba(19,34,56,0.8)', zeroline=False),
    legend=dict(orientation='h', y=1.08, x=0.5, xanchor='center', bgcolor='rgba(0,0,0,0)')
)

fig2.show()

# Print summary statistics
avg_prem = df_karat_prices['prem_pct_24k'].mean()
max_prem = df_karat_prices['prem_pct_24k'].max()
max_prem_date = df_karat_prices['prem_pct_24k'].idxmax().strftime('%b %Y')
print(f'\n=== Inflation Premium Summary (24K) ===')
print(f'Average Premium %:     {avg_prem:.1f}%')
print(f'Peak Premium %:        {max_prem:.1f}% in {max_prem_date}')
print(f'Current Premium %:     {df_karat_prices["prem_pct_24k"].iloc[-1]:.1f}%')


=== Inflation Premium Summary (24K) ===
Average Premium %:     35.5%
Peak Premium %:        71.3% in Apr 2026
Current Premium %:     68.5%


In [10]:
# ─── Chart 3: Net Portfolio Performance - All Asset Classes ──────────────────

KARAT_COLORS = {'24K': '#FFF9C4', '21K': '#FFD700', '18K': '#CD7F32'}

fig3 = go.Figure()

fig3.add_trace(go.Scatter(
    x=df_portfolio.index, y=df_portfolio['port_24k_net'],
    name='24K Gold (net of making charges & spread)',
    line=dict(color=KARAT_COLORS['24K'], width=2.2),
    hovertemplate='24K: %{y:,.0f} EGP<extra></extra>'
))

fig3.add_trace(go.Scatter(
    x=df_portfolio.index, y=df_portfolio['port_21k_net'],
    name='21K Gold (net)',
    line=dict(color=KARAT_COLORS['21K'], width=2.2),
    hovertemplate='21K: %{y:,.0f} EGP<extra></extra>'
))

fig3.add_trace(go.Scatter(
    x=df_portfolio.index, y=df_portfolio['port_18k_net'],
    name='18K Gold (net)',
    line=dict(color=KARAT_COLORS['18K'], width=2.2),
    hovertemplate='18K: %{y:,.0f} EGP<extra></extra>'
))

fig3.add_trace(go.Scatter(
    x=df_portfolio.index, y=df_portfolio['port_usd'],
    name='USD (converted to EGP)',
    line=dict(color='#4CC9F0', width=1.8, dash='dot'),
    hovertemplate='USD: %{y:,.0f} EGP<extra></extra>'
))

fig3.add_trace(go.Scatter(
    x=df_portfolio.index, y=df_portfolio['port_cash'],
    name='EGP Cash (no return)',
    line=dict(color='#2A3A55', width=1.2, dash='dot'),
    hovertemplate='Cash: %{y:,.0f} EGP<extra></extra>'
))

add_crisis_overlays(fig3, df_portfolio.index)

fig3.update_layout(
    **PLOTLY_BASE,
    height=520,
    title=dict(
        text='Net Return Portfolio Simulation \u2014 100,000 EGP Invested Jan 2020 | Friction-Adjusted',
        font=dict(size=13, color='#FFD700'), x=0.5
    ),
    xaxis=dict(title='Date', gridcolor='rgba(19,34,56,0.8)', zeroline=False),
    yaxis=dict(title='Portfolio Value (EGP)', gridcolor='rgba(19,34,56,0.8)', zeroline=False),
    legend=dict(orientation='h', y=1.10, x=0.5, xanchor='center', bgcolor='rgba(0,0,0,0)')
)

fig3.show()

# Print comparative performance summary
print('\n=== Net Return Performance Summary (EGP 100,000 baseline) ===')
for karat, col in [('24K', 'port_24k_net'), ('21K', 'port_21k_net'), ('18K', 'port_18k_net')]:
    final_val = df_portfolio[col].iloc[-1]
    total_ret = (final_val / 100_000 - 1) * 100
    daily_ret = df_portfolio[col].pct_change().dropna()
    sharpe    = (daily_ret.mean() * 252) / (daily_ret.std() * np.sqrt(252))
    max_dd    = ((df_portfolio[col] / df_portfolio[col].cummax()) - 1).min() * 100
    print(f'  {karat}: Final={final_val:,.0f} EGP | Return={total_ret:+.1f}% | Sharpe={sharpe:.2f} | MaxDD={max_dd:.1f}%')

usd_final = df_portfolio['port_usd'].iloc[-1]
usd_ret   = (usd_final / 100_000 - 1) * 100
print(f'  USD: Final={usd_final:,.0f} EGP | Return={usd_ret:+.1f}%')
print(f'  Cash: Return=0.0% (nominal) | Real return: deeply negative vs inflation')


=== Net Return Performance Summary (EGP 100,000 baseline) ===
  24K: Final=805,131 EGP | Return=+705.1% | Sharpe=1.14 | MaxDD=-23.4%
  21K: Final=719,273 EGP | Return=+619.3% | Sharpe=1.14 | MaxDD=-23.2%
  18K: Final=645,862 EGP | Return=+545.9% | Sharpe=1.13 | MaxDD=-22.9%
  USD: Final=311,603 EGP | Return=+211.6%
  Cash: Return=0.0% (nominal) | Real return: deeply negative vs inflation


In [11]:
# ─── Chart 4: Technical Indicators Dashboard - RSI, MACD, Bollinger Bands ────

fig4 = make_subplots(
    rows=3, cols=1, shared_xaxes=True,
    row_heights=[0.55, 0.25, 0.20],
    vertical_spacing=0.07,
    subplot_titles=[
        'Gold Price 24K + Bollinger Bands (20-day, 2\u03c3)',
        'MACD (12, 26, 9) \u2014 Momentum Signal',
        'RSI-14 \u2014 Overbought / Oversold'
    ]
)

# ── Bollinger Bands ────────────────────────────────────────────────────────────
fig4.add_trace(go.Scatter(
    x=df_technical.index, y=df_technical['bb_upper_24k'],
    line=dict(width=0), showlegend=False
), row=1, col=1)

fig4.add_trace(go.Scatter(
    x=df_technical.index, y=df_technical['bb_lower_24k'],
    fill='tonexty', fillcolor='rgba(180,180,255,0.05)',
    line=dict(width=0), name='Bollinger Bands'
), row=1, col=1)

fig4.add_trace(go.Scatter(
    x=df_karat_prices.index, y=df_karat_prices['price_24k'],
    name='Price 24K', line=dict(color='#D8E4F0', width=1.8),
    hovertemplate='%{x|%d %b %Y}: %{y:,.0f} EGP<extra></extra>'
), row=1, col=1)

fig4.add_trace(go.Scatter(
    x=df_technical.index, y=df_technical['sma50_24k'],
    name='SMA-50', line=dict(color='#FF9F43', width=1.1, dash='dot')
), row=1, col=1)

fig4.add_trace(go.Scatter(
    x=df_technical.index, y=df_technical['sma200_24k'],
    name='SMA-200', line=dict(color='#A855F7', width=1.1, dash='dot')
), row=1, col=1)

# ── BUY / SELL markers ─────────────────────────────────────────────────────────
buy_mask  = df_technical['signal_24k'] == 'BUY'
sell_mask = df_technical['signal_24k'] == 'SELL'

fig4.add_trace(go.Scatter(
    x=df_karat_prices.index[buy_mask], y=df_karat_prices['price_24k'][buy_mask],
    mode='markers', name='BUY \u25b2',
    marker=dict(symbol='triangle-up', color='#06D6A0', size=7, line=dict(width=1, color='white'))
), row=1, col=1)

fig4.add_trace(go.Scatter(
    x=df_karat_prices.index[sell_mask], y=df_karat_prices['price_24k'][sell_mask],
    mode='markers', name='SELL \u25bc',
    marker=dict(symbol='triangle-down', color='#EF476F', size=7, line=dict(width=1, color='white'))
), row=1, col=1)

# ── MACD ───────────────────────────────────────────────────────────────────────
hist_colors = [
    '#06D6A0' if v >= 0 else '#EF476F'
    for v in df_technical['macd_hist_24k']
]
fig4.add_trace(go.Bar(
    x=df_technical.index, y=df_technical['macd_hist_24k'],
    name='MACD Histogram', marker_color=hist_colors, opacity=0.7
), row=2, col=1)

fig4.add_trace(go.Scatter(
    x=df_technical.index, y=df_technical['macd_24k'],
    name='MACD Line', line=dict(color='#4CC9F0', width=1.5)
), row=2, col=1)

fig4.add_trace(go.Scatter(
    x=df_technical.index, y=df_technical['macd_sig_24k'],
    name='Signal Line', line=dict(color='#FF9F43', width=1.5)
), row=2, col=1)

# ── RSI-14 ─────────────────────────────────────────────────────────────────────
fig4.add_trace(go.Scatter(
    x=df_technical.index, y=df_technical['rsi_24k'],
    name='RSI-14', line=dict(color='#A855F7', width=1.7)
), row=3, col=1)

fig4.add_hline(y=70, line_dash='dot', line_color='#EF476F', opacity=0.45, row=3, col=1)
fig4.add_hline(y=30, line_dash='dot', line_color='#06D6A0', opacity=0.45, row=3, col=1)
fig4.add_hrect(y0=70, y1=100, fillcolor='rgba(239,71,111,0.04)', line_width=0, row=3, col=1)
fig4.add_hrect(y0=0,  y1=30,  fillcolor='rgba(6,214,160,0.04)',  line_width=0, row=3, col=1)

add_crisis_overlays(fig4, df_technical.index, rows=[1, 2, 3], y_ann=0.93)

fig4.update_layout(
    **PLOTLY_BASE,
    height=860,
    title=dict(
        text='Technical Indicator Dashboard \u2014 24K Gold | BUY/HOLD/SELL Signal Engine',
        font=dict(size=13, color='#FFD700'), x=0.5
    ),
    legend=dict(orientation='h', y=1.04, x=0.5, xanchor='center', bgcolor='rgba(0,0,0,0)', font=dict(size=9.5))
)

fig4.update_yaxes(title_text='EGP/gram', row=1, col=1)
fig4.update_yaxes(title_text='MACD',     row=2, col=1)
fig4.update_yaxes(title_text='RSI',      range=[0, 100], row=3, col=1)

fig4.show()

# Print current signal summary
current_sig  = df_technical['signal_24k'].iloc[-1]
current_rsi  = df_technical['rsi_24k'].iloc[-1]
current_macd = df_technical['macd_24k'].iloc[-1]
print(f'\n=== Current Technical Signal Summary (24K) ===')
print(f'  Composite Signal: {current_sig}')
print(f'  RSI-14:           {current_rsi:.1f}')
print(f'  MACD:             {current_macd:,.2f}')
print(f'  BB Upper:         {df_technical["bb_upper_24k"].iloc[-1]:,.0f} EGP')
print(f'  BB Lower:         {df_technical["bb_lower_24k"].iloc[-1]:,.0f} EGP')


=== Current Technical Signal Summary (24K) ===
  Composite Signal: BUY
  RSI-14:           27.8
  MACD:             -237.86
  BB Upper:         7,913 EGP
  BB Lower:         6,688 EGP


In [12]:
# ─── Chart 5: Correlation Heatmap - Gold vs Macroeconomic Drivers ────────────

corr_df = df_gold_prices[[
    'gold_usd_oz', 'usd_egp_official', 'crude_oil',
    'us_10y_treasury', 'sp500'
]].dropna().corr()

labels = ['Gold USD/oz', 'USD/EGP Rate', 'Crude Oil', 'US 10Y Treasury', 'S&P 500']
corr_df.columns = labels
corr_df.index   = labels

fig5 = px.imshow(
    corr_df,
    text_auto='.2f',
    color_continuous_scale=[[0, '#EF476F'], [0.5, '#060C18'], [1, '#06D6A0']],
    zmin=-1, zmax=1,
    title='Pearson Correlation Matrix \u2014 Gold vs Macroeconomic Drivers (Jan 2020\u2013Present)'
)

fig5.update_layout(
    **PLOTLY_BASE,
    height=480,
    coloraxis_showscale=True,
    title=dict(font=dict(size=13, color='#FFD700'), x=0.5)
)
fig5.update_traces(textfont=dict(size=11, family='Arial'))
fig5.show()

# Print correlation with gold
print('\n=== Correlations with Gold USD/oz ===')
gold_corr = corr_df['Gold USD/oz'].drop('Gold USD/oz')
for driver, corr_val in gold_corr.sort_values(ascending=False).items():
    direction = 'positive' if corr_val > 0 else 'negative'
    print(f'  {driver:<22}: {corr_val:+.3f} ({direction})')


=== Correlations with Gold USD/oz ===
  S&P 500               : +0.890 (positive)
  USD/EGP Rate          : +0.778 (positive)
  US 10Y Treasury       : +0.542 (positive)
  Crude Oil             : +0.079 (positive)


---
## Cell 8 - Pipeline Summary

Full execution summary and data quality report for all four output tables.

In [13]:
print('=' * 65)
print('   GOLD EGYPT ETL PIPELINE \u2014 EXECUTION SUMMARY')
print('   DEPI Final Project | Mahmoud Shamoun | 2026')
print('=' * 65)

date_min = df_gold_prices.index.min().strftime('%Y-%m-%d')
date_max = df_gold_prices.index.max().strftime('%Y-%m-%d')

print(f'\n  Data Window:          {date_min}  \u2192  {date_max}')
print(f'  Total Trading Days:   {len(df_gold_prices):,}')

print('\n  TABLE READINESS REPORT')
print(f'  {"Table":<30} {"Rows":>8}  {"NaN Cells":>10}')
print('  ' + '-' * 52)

for label, df_check in [
    ('gold_prices',          df_gold_prices),
    ('karat_prices',         df_karat_prices),
    ('technical_indicators', df_technical),
    ('portfolio_simulation', df_portfolio),
    ('crisis_events',        CRISIS_EVENTS_DF),
]:
    null_count = df_check.isnull().sum().sum()
    print(f'  {label:<30} {len(df_check):>8,}  {null_count:>10,}')

print()
print('  CURRENT MARKET SNAPSHOT')
print(f'  Gold USD/oz:          ${df_gold_prices["gold_usd_oz"].iloc[-1]:>10,.2f}')
print(f'  USD/EGP Rate:          {df_gold_prices["usd_egp_official"].iloc[-1]:>10,.4f}')
print(f'  Crude Oil (USD/bbl):   {df_gold_prices["crude_oil"].iloc[-1]:>10,.2f}')
print(f'  US 10Y Treasury (%):   {df_gold_prices["us_10y_treasury"].iloc[-1]:>10,.4f}')
print(f'  S&P 500:               {df_gold_prices["sp500"].iloc[-1]:>10,.2f}')
print()
print(f'  24K Price (EGP/gram):  {df_karat_prices["price_24k"].iloc[-1]:>10,.2f}')
print(f'  21K Price (EGP/gram):  {df_karat_prices["price_21k"].iloc[-1]:>10,.2f}')
print(f'  18K Price (EGP/gram):  {df_karat_prices["price_18k"].iloc[-1]:>10,.2f}')
print(f'  Inflation Premium 24K: {df_karat_prices["prem_pct_24k"].iloc[-1]:>9,.1f}%')
print(f'  Current Signal (24K):  {df_technical["signal_24k"].iloc[-1]:>10}')
print()
print('  STATUS: Pipeline complete. Ready to execute load_to_mysql().')
print('=' * 65)

logger.info('Pipeline execution complete. All cells ran successfully.')

2026-06-22 22:40:31,838 - INFO - Pipeline execution complete. All cells ran successfully.


   GOLD EGYPT ETL PIPELINE — EXECUTION SUMMARY
   DEPI Final Project | Mahmoud Shamoun | 2026

  Data Window:          2020-01-01  →  2026-06-19
  Total Trading Days:   1,684

  TABLE READINESS REPORT
  Table                              Rows   NaN Cells
  ----------------------------------------------------
  gold_prices                       1,684           0
  karat_prices                      1,684           0
  technical_indicators              1,684          70
  portfolio_simulation              1,684           0
  crisis_events                        12           0

  CURRENT MARKET SNAPSHOT
  Gold USD/oz:          $  4,224.10
  USD/EGP Rate:             49.8948
  Crude Oil (USD/bbl):        76.60
  US 10Y Treasury (%):       4.4510
  S&P 500:                 7,500.58

  24K Price (EGP/gram):    6,836.11
  21K Price (EGP/gram):    6,079.10
  18K Price (EGP/gram):    5,302.08
  Inflation Premium 24K:      68.5%
  Current Signal (24K):         BUY

  STATUS: Pipeline complete. Re